# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mursaleen-developer/fly-rank-ML-Intenship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Unit of analysis

One row represents **one content item for one client on one report date**.

The grain is:

**client × content item × day**

I will use the `fact_content_daily_performance` table for this lane.

### Time window

I will use **March 2026** (`2026-03-01` through `2026-03-31`) as the development and verification month.

I chose March 2026 because it is a mid-panel month. I will not use the final June 2026 sample while developing the label or feature logic.

In [11]:
from huggingface_hub import hf_hub_download
import pandas as pd

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=hf_token
)

march_df = pd.read_parquet(march_path)

print("March 2026 shape:", march_df.shape)

print("\nColumns:")
for column in march_df.columns:
    print("-", column)

print("\nFirst 5 rows:")
display(march_df.head())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

March 2026 shape: (9841378, 30)

Columns:
- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other
- scroll_events

First 5 rows:


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Fields: feature / label / context / excluded

### Features

The features for the March decision point will come from `fact_content_daily_performance`.

I will use signals that are available during March 2026, such as:

- `gsc_impressions` — search visibility available during the decision window.
- `gsc_clicks` — search clicks available during the decision window.
- `gsc_avg_position` — average search position available during the decision window.
- `ga4_sessions` — website sessions available during the decision window.
- `sessions_organic` — organic sessions available during the decision window.

### Label

The label will represent a **future decline outcome** after the March decision point.

I will derive the label from the later April–June 2026 query-performance window rather than using future performance as a feature.

A content item will be labelled as declining when its later-period search impressions show a decline from the previous 30-day period.

### Context

These fields help describe whether the data is available and identify the observation:

- `report_date`
- `client_hash_id`
- `content_hash_id`
- `client_has_gsc`
- `client_has_ga4`
- `gsc_data_available`
- `ga4_data_available`

### Excluded

I will exclude:

- `trend_direction` or any equivalent label-derived field, because it would directly reveal the outcome.
- Future April–June performance fields, because they would not be available at the March decision moment.
- Identifier fields such as `client_hash_id` and `content_hash_id` as predictive features, because they identify entities rather than describe their content/performance.

In [13]:
# Inspect the 90-day search query table

query_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_query_90d.parquet",
    repo_type="dataset",
    token=hf_token
)

query_df = pd.read_parquet(query_path)

print("fact_content_query_90d shape:", query_df.shape)

print("\nColumns:")
for col in query_df.columns:
    print("-", col)

print("\nFirst 5 rows:")
display(query_df.head())


fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

fact_content_query_90d shape: (2414248, 21)

Columns:
- client_hash_id
- content_hash_id
- query_hash_id
- query_char_count
- query_token_count
- window_start
- window_end
- impressions_90d
- clicks_90d
- impressions_last30
- clicks_last30
- impressions_prev30
- clicks_prev30
- avg_position_90d
- avg_position_last30
- avg_position_prev30
- content_total_impressions_90d
- content_visible_query_count
- rare_query_count
- rare_impressions_share
- anonymized_impressions_share

First 5 rows:


,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,16,2,2026-04-02,2026-06-30,16,0,11,...,5,0,23.562500,24.272727,22.000000,1466,14,32,0.043656,0.725102
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,24,4,2026-04-02,2026-06-30,55,0,1,...,1,0,2.200000,13.000000,0.000000,1466,14,32,0.043656,0.725102
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,18,3,2026-04-02,2026-06-30,14,0,0,...,0,0,3.428571,NaN,NaN,1466,14,32,0.043656,0.725102


### Verification queries

I use March 2026 as the mid-panel development month.

The three checks below verify:

1. the row grain,
2. the row count and date span,
3. data availability using `IS TRUE`.
### Five features

I will use five features from the March 2026 daily performance data.

1. `gsc_impressions` — available at the decision moment because March Search Console impressions are already observed.
2. `gsc_clicks` — available at the decision moment because March Search Console clicks are already observed.
3. `gsc_avg_position` — available at the decision moment because the March average search position is already observed.
4. `ga4_sessions` — available at the decision moment because March Analytics session data is already observed.
5. `sessions_organic` — available at the decision moment because March organic session data is already observed.

These features describe search visibility and website performance without using the future outcome.

### Leakage trap

To demonstrate data leakage, I intentionally add a label-derived column to the feature frame.

This column contains information derived from the outcome, so it would not be available at the decision moment.

I expect the quick score to become unrealistically strong when this leaked column is included.

After demonstrating the leakage, I will remove the column and keep the honest feature set.

In [17]:
# Query 1: Verify the grain
# One row should represent one client × content item × report date.

grain_check = (
    march_df
    .groupby(["client_hash_id", "content_hash_id", "report_date"])
    .size()
)

print("Total rows:", len(march_df))
print("Unique client × content × date combinations:", grain_check.size)
print("Duplicate grain combinations:", (grain_check > 1).sum())

# Query 2: Verify row count and date span

print("March 2026 row count:", len(march_df))
print("Minimum report date:", march_df["report_date"].min())
print("Maximum report date:", march_df["report_date"].max())

# Query 3: Availability check
# Equivalent to SQL: WHERE gsc_data_available IS TRUE

available_df = march_df[
    march_df["gsc_data_available"].eq(True)
]

print("Rows with gsc_data_available IS TRUE:", len(available_df))
print(
    "Percentage of March rows available:",
    round(len(available_df) / len(march_df) * 100, 2),
    "%"
)


Total rows: 9841378
Unique client × content × date combinations: 9841378
Duplicate grain combinations: 0
March 2026 row count: 9841378
Minimum report date: 2026-03-01
Maximum report date: 2026-03-31
Rows with gsc_data_available IS TRUE: 3611061
Percentage of March rows available: 36.69 %


In [18]:
# Build the five-feature frame

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "sessions_organic"
]

feature_df = march_df[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date"
    ] + feature_cols
].copy()

print("Feature frame shape:", feature_df.shape)

display(feature_df.head(10))

Feature frame shape: (9841378, 8)


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,sessions_organic
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,NaN,NaN
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,NaN,NaN
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,NaN,NaN
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,NaN,NaN
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,NaN,NaN
5,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,1,7.347280,NaN,NaN
6,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,0,7.832461,NaN,NaN
7,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,55,0,3.272727,NaN,NaN
8,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,77,0,5.636364,NaN,NaN
9,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01,2,0,4.500000,NaN,NaN


In [21]:
# Create a real future outcome label from the April-June query window

query_df["future_decline_label"] = (
    query_df["impressions_last30"] < query_df["impressions_prev30"]
).astype(int)

print("Future label created.")

print("\nFuture label distribution:")
print(query_df["future_decline_label"].value_counts())

print("\nFuture label percentages:")
print(
    query_df["future_decline_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Future label created.

Future label distribution:
future_decline_label
1    1373407
0    1040841
Name: count, dtype: int64

Future label percentages:
future_decline_label
1    56.89
0    43.11
Name: proportion, dtype: float64


In [22]:
# Aggregate the future query outcome to client × content level

future_content_label = (
    query_df
    .groupby(["client_hash_id", "content_hash_id"])["future_decline_label"]
    .mean()
    .reset_index()
)

# Convert to a binary content-level label:
# 1 if more than half of the observed queries declined
future_content_label["future_decline_label"] = (
    future_content_label["future_decline_label"] >= 0.5
).astype(int)

print("Content-level future label shape:", future_content_label.shape)

print("\nLabel distribution:")
print(future_content_label["future_decline_label"].value_counts())

display(future_content_label.head())


Content-level future label shape: (133852, 3)

Label distribution:
future_decline_label
1    79903
0    53949
Name: count, dtype: int64


,client_hash_id,content_hash_id,future_decline_label
0,client_06d356715a8ff3b6,content_0058bd88fb1821f2,0
1,client_06d356715a8ff3b6,content_0059a4d4195810c9,0
2,client_06d356715a8ff3b6,content_005b6b7f7b8dda7f,1
3,client_06d356715a8ff3b6,content_0094c7d0fbcc07b7,0
4,client_06d356715a8ff3b6,content_00a34394d4ee05ce,1


In [23]:
# Join the future outcome to the March feature frame

leakage_df = feature_df.merge(
    future_content_label,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Leakage dataframe shape:", leakage_df.shape)

print("\nFuture label distribution:")
print(leakage_df["future_decline_label"].value_counts())

display(
    leakage_df[
        [
            "client_hash_id",
            "content_hash_id",
            "report_date",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "ga4_sessions",
            "sessions_organic",
            "future_decline_label"
        ]
    ].head(10)
)

Leakage dataframe shape: (3224352, 10)

Future label distribution:
future_decline_label
1    2081294
0    1143058
Name: count, dtype: int64


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,sessions_organic,future_decline_label
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,NaN,NaN,1
1,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,NaN,NaN,1
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,1,7.347280,NaN,NaN,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,0,7.832461,NaN,NaN,1
4,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,55,0,3.272727,NaN,NaN,0
5,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,77,0,5.636364,NaN,NaN,1
6,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01,2,0,4.500000,NaN,NaN,0
7,client_73cda7b4e4f265ea,content_1855a661b4d36130,2026-03-01,14,0,3.428571,NaN,NaN,0
8,client_73cda7b4e4f265ea,content_712c365258cee05c,2026-03-01,223,0,3.892377,NaN,NaN,1
9,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,2026-03-01,6,0,4.500000,NaN,NaN,1


In [24]:
# Deliberate leakage demonstration
# We incorrectly use the future label itself as the priority score.

leakage_df["leaked_priority_score"] = leakage_df["future_decline_label"]

k = 50

top_k = (
    leakage_df
    .sort_values("leaked_priority_score", ascending=False)
    .head(k)
)

precision_at_50_leaked = top_k["future_decline_label"].mean()

print("Leaked Precision@50:", round(precision_at_50_leaked, 4))
print("Leaked Precision@50 (%):", round(precision_at_50_leaked * 100, 2), "%")


Leaked Precision@50: 1.0
Leaked Precision@50 (%): 100.0 %


In [25]:
# Remove the leaked column and keep only the honest features

honest_feature_cols = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "sessions_organic"
]

honest_feature_df = leakage_df[honest_feature_cols].copy()

print("Leaked column present:", "future_decline_label" in honest_feature_df.columns)
print("Honest feature frame shape:", honest_feature_df.shape)

display(honest_feature_df.head())

Leaked column present: False
Honest feature frame shape: (3224352, 8)


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,sessions_organic
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,NaN,NaN
1,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,NaN,NaN
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,1,7.347280,NaN,NaN
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,0,7.832461,NaN,NaN
4,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,55,0,3.272727,NaN,NaN


### Limitation of this data

This data has several limitations.

1. **Incomplete data availability:** Only 36.69% of March rows have `gsc_data_available IS TRUE`, so GSC-based features are not available for every row.

2. **Limited historical context:** The March slice alone cannot tell us the long-term history of a content page or fully explain why its performance changed.

3. **Overlapping time windows:** The 90-day query table contains overlapping time periods, so the last-30-day and previous-30-day metrics are related parts of the same observation window.

4. **No explanation of causality:** The data can show that a page's performance changed, but it cannot tell us the exact reason. For example, a decline could be caused by seasonality, competition, search algorithm changes, or technical issues.

5. **Decision-support limitation:** The data can help rank pages for review, but it cannot determine whether a page definitely needs a refresh or what changes an editor should make.

In [26]:
# Check the main data limitations in the March 2026 slice

total_rows = len(march_df)

gsc_available_rows = march_df["gsc_data_available"].eq(True).sum()

print("Total March rows:", total_rows)
print("Rows with GSC data available:", gsc_available_rows)
print(
    "GSC availability:",
    round(gsc_available_rows / total_rows * 100, 2),
    "%"
)

print("\nDate range:")
print("Minimum:", march_df["report_date"].min())
print("Maximum:", march_df["report_date"].max())


Total March rows: 9841378
Rows with GSC data available: 3611061
GSC availability: 36.69 %

Date range:
Minimum: 2026-03-01
Maximum: 2026-03-31


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.